# Who Wants to be a PoliMillionaire? — Group Assignment

**Course:** Natural Language Processing, Politecnico di Milano, AY 2025/26  
**Due:** 2 June 2026, 23:00 (WeBeep)  
**Repository:** https://github.com/leonfuss/polimillionaire

## Team

| Name | Email | GitHub | PoliMillionaire username |
|---|---|---|---|
| _name 1_ | _email_ | _gh handle_ | _username_ |
| _name 2_ | _email_ | _gh handle_ | _username_ |
| _name 3_ | _email_ | _gh handle_ | _username_ |
| _name 4_ | _email_ | _gh handle_ | _username_ |
| _name 5_ | _email_ | _gh handle_ | _username_ |

**Video:** _link to be added before submission_

## Use of coding assistants

_Edit before submission. Suggested template:_

> During this project we used <tool name(s)> for: scaffolding boilerplate, generating docstrings, debugging stack traces, and suggesting prompt variants. The model architectures, evaluation methodology, and final analyses were designed by us; we reviewed and edited every line of generated code before committing it. The assignment was not given as a whole to any LLM.

## Colab setup

Run this once at the top of a fresh Colab runtime. Re-running the cell on the same runtime is fine — if the repo is already cloned it pulls latest `main` instead. Requires `GH_TOKEN` in Colab Secrets (fine-grained PAT, Contents read on this repo) — see the README for setup.

Two things in here are load-bearing: the `--no-deps` on the editable install stops pip from re-resolving torch/transformers, and the `sys.path` line at the end works around a hatchling/Colab quirk where editable install hooks (PEP 660 `.pth` files) only fire at Python startup, after the kernel is already up.

In [ ]:
import os
import sys

from google.colab import userdata

gh_token = userdata.get("GH_TOKEN")
repo_dir = "/content/polimillionaire"

if os.path.isdir(repo_dir):
    os.chdir(repo_dir)
    !git checkout -q main && git pull --ff-only origin main
else:
    !git clone https://{gh_token}@github.com/leonfuss/polimillionaire.git $repo_dir
    os.chdir(repo_dir)

# CUDA wheel index for llama-cpp-python's pre-built T4 wheel.
!pip install -q -r requirements-colab.txt \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
    && pip install -q -e . --no-deps

if "/content/polimillionaire/src" not in sys.path:
    sys.path.insert(0, "/content/polimillionaire/src")

## Mount the shared question log

The question log is the canonical artefact of the project — every observed question + our predictions. It lives in a shared Google Drive folder so all five teammates write to the same DB. Set `POLIMILLIONAIRE_DB_PATH` in Colab Secrets if you've moved it; otherwise the cell below uses the default path.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
os.environ.setdefault(
    "POLIMILLIONAIRE_DB_PATH",
    "/content/drive/MyDrive/PoliMillionaire/questions.sqlite",
)
print("DB path:", os.environ["POLIMILLIONAIRE_DB_PATH"])

## Corpus bootstrap — manual play

Each teammate is assigned one competition. Run a few games on yours; every question + your answer lands in the shared DB. Type the option id at each prompt, or `q` to abort cleanly.

In [ ]:
from polimillionaire import make_client
from polimillionaire.play import manual_play_loop

client = make_client()
manual_play_loop(client, competition_id=0, max_games=3)  # change competition_id

## LLM smoke test

Loads the default model (Qwen3-8B Q4_K_M, ~5 GB) from Hugging Face. First run downloads the GGUF (~1–2 minutes on Colab); subsequent runs in the same session are instant. Confirms the GBNF JSON-mode path works end-to-end.

In [ ]:
from polimillionaire import load_llm

llm = load_llm("qwen3-8b")

schema = {
    "type": "object",
    "properties": {
        "answer": {"type": "string", "enum": ["A", "B", "C", "D"]},
        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
        "rationale": {"type": "string"},
    },
    "required": ["answer", "confidence", "rationale"],
}

messages = [
    {"role": "system", "content": "You answer multiple-choice trivia. Output JSON."},
    {
        "role": "user",
        "content": "Q: What is the capital of France?\nA) Berlin\nB) Madrid\nC) Paris\nD) Rome",
    },
]

print(llm.complete_json(messages, schema))